In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance.liquidity.assumption_reference (
framework STRING COMMENT 'Framework, e.g. Y15',
assumption_category STRING COMMENT 'Deposit, Product, Behavioral, Regulatory, Classification',
assumption_code STRING COMMENT 'Stable business key, e.g. Y15_DEP_CLASS',
assumption_name STRING COMMENT 'Business name of assumption',
assumption_value STRING COMMENT 'Value assigned to assumption',
assumption_description STRING COMMENT 'Explanation of assumption',
effective_date DATE,
expiration_date DATE,
active_flag STRING COMMENT 'Y/N',
source_system STRING,
owner STRING,
comments STRING,
created_by STRING,
created_ts TIMESTAMP,
modified_by STRING,
modified_ts TIMESTAMP
)
USING DELTA;
DESCRIBE finance.liquidity.assumption_reference;



In [0]:
%sql
CREATE TABLE IF NOT EXISTS finance.liquidity.assumption_reference_bkp_y15_20261102 AS
SELECT * FROM finance.liquidity.assumption_reference WHERE framework = 'Y15';
CREATE OR REPLACE TABLE finance.liquidity.assumption_reference_stg_y15 AS
SELECT
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
QUALIFY ROW_NUMBER() OVER (
PARTITION BY assumption_code, effective_date
ORDER BY modified_ts DESC, created_ts DESC
) = 1;
DELETE FROM finance.liquidity.assumption_reference WHERE framework = 'Y15';
INSERT INTO finance.liquidity.assumption_reference (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
SELECT
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
FROM finance.liquidity.assumption_reference_stg_y15;
DROP TABLE IF EXISTS finance.liquidity.assumption_reference_stg_y15;
SELECT assumption_code, effective_date, COUNT(1) AS row_count
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
GROUP BY assumption_code, effective_date
ORDER BY assumption_code;


In [0]:
%sql
MERGE INTO finance.liquidity.assumption_reference AS tgt
USING (
SELECT * FROM VALUES
('Y15', 'Classification', 'Y15_DEP_CLASS', 'Deposit Classification', 'Retail',
'Y15 Product Classification', DATE'2026-10-28', 'Treasury', 'Treasury')
AS s(framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, source_system, owner)
) AS src
ON tgt.framework = src.framework
AND tgt.assumption_code = src.assumption_code
AND tgt.effective_date = src.effective_date
WHEN NOT MATCHED THEN INSERT (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
VALUES (
src.framework, src.assumption_category, src.assumption_code, src.assumption_name, src.assumption_value,
src.assumption_description, src.effective_date, NULL, 'Y', src.source_system,
src.owner, NULL, current_user(), current_timestamp(), current_user(), current_timestamp()
);
SELECT assumption_code, assumption_name, assumption_value, effective_date, active_flag
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
ORDER BY assumption_code, effective_date DESC;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW y15_assumption_change AS
SELECT * FROM VALUES
('Y15_DEP_CLASS', 'Retail', DATE'2026-11-02', 'Describe reason for change')
AS c(assumption_code, new_value, new_effective_date, change_comment);
-- Step 1: expire the current active row
MERGE INTO finance.liquidity.assumption_reference AS tgt
USING y15_assumption_change AS c
ON tgt.framework = 'Y15'
AND tgt.assumption_code = c.assumption_code
AND tgt.active_flag = 'Y'
WHEN MATCHED AND tgt.assumption_value <> c.new_value THEN UPDATE SET
tgt.active_flag = 'N',
tgt.expiration_date = date_sub(c.new_effective_date, 1),
tgt.modified_by = current_user(),
tgt.modified_ts = current_timestamp();
-- Step 2: insert the new active row (copies other attributes from the latest row)
INSERT INTO finance.liquidity.assumption_reference (
framework, assumption_category, assumption_code, assumption_name, assumption_value,
assumption_description, effective_date, expiration_date, active_flag, source_system,
owner, comments, created_by, created_ts, modified_by, modified_ts
)
SELECT
p.framework, p.assumption_category, p.assumption_code, p.assumption_name, c.new_value,
p.assumption_description, c.new_effective_date, CAST(NULL AS DATE), 'Y', p.source_system,
p.owner, c.change_comment, current_user(), current_timestamp(), current_user(), current_timestamp()
FROM y15_assumption_change AS c
JOIN (
SELECT *
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
QUALIFY ROW_NUMBER() OVER (
PARTITION BY assumption_code
ORDER BY effective_date DESC, modified_ts DESC
) = 1
) AS p
ON p.assumption_code = c.assumption_code
WHERE NOT EXISTS (
SELECT 1
FROM finance.liquidity.assumption_reference x
WHERE x.framework = 'Y15'
AND x.assumption_code = c.assumption_code
AND x.active_flag = 'Y'
);
SELECT assumption_code, assumption_value, effective_date, expiration_date, active_flag, comments
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
AND assumption_code IN (SELECT assumption_code FROM y15_assumption_change)
ORDER BY effective_date DESC;


In [0]:
%sql
CREATE OR REPLACE VIEW finance.liquidity.v_y15_assumptions_current AS
SELECT
framework, assumption_category, assumption_code, assumption_name,
assumption_value, assumption_description, effective_date, owner, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
AND active_flag = 'Y'
AND effective_date <= current_date()
AND (expiration_date IS NULL OR expiration_date >= current_date());
WITH y15 AS (
SELECT * FROM finance.liquidity.assumption_reference WHERE framework = 'Y15'
),
checks AS (
SELECT 1 AS check_order, 'Duplicate active assumption rows' AS check_name, COUNT(1) AS issues
FROM (SELECT assumption_code FROM y15 WHERE active_flag = 'Y'
GROUP BY assumption_code HAVING COUNT(1) > 1)
UNION ALL
SELECT 2, 'Assumption codes with no active row', COUNT(1)
FROM (SELECT assumption_code FROM y15 GROUP BY assumption_code
HAVING SUM(CASE WHEN active_flag = 'Y' THEN 1 ELSE 0 END) = 0)
UNION ALL
SELECT 3, 'Active rows with blank value', COUNT_IF(TRIM(COALESCE(assumption_value, '')) = '')
FROM y15 WHERE active_flag = 'Y'
)
SELECT
check_name,
issues,
CASE WHEN issues = 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM checks
ORDER BY check_order;


In [0]:
%sql
SELECT
assumption_code, assumption_name, assumption_value, effective_date,
expiration_date, active_flag, comments, modified_by, modified_ts
FROM finance.liquidity.assumption_reference
WHERE framework = 'Y15'
ORDER BY assumption_code, effective_date DESC;


In [0]:
display(_sqldf)



--- 

     `finance.liquidity.y15_SchG_source_assumptions`
            ----- (Tier in the schedule)
            ----- (Line in the actual schedule)
            ----- (Weigting Applied)
            ----- (Present in Schedule)
            ----- y15_SchG rollup
            ----- y15_SchG class
            ----- y15_SchG class code (Vlad's CCID)
            ----- balance_type (ledger_balance, nonop_balance,op_balance) 
            ----- Maturity Bucket Long
            ----- Maturity Column (A-E)
            ----- FseFlag
            ----- WholesaleFlag,
            ----- Counterparty_Type,
            ----- IcsFlag,
            ----- BrokeredFlag
            ----- Ops_flag
            ----- CollateralFlag
            ----- DepositType
            ----- ProductType
            ----- internal_elim_flag
            ----- BdspFlag
            ----- legacy_bank
            ----- SourceTable (deposits_plus, wholesalefunding)

            ----- assumption_effective_date
            ----- assumption_end_date
            ----- approved flag
            ----- entered by _______
            ----- timestamp here ... etc





     `finance.liquidity.y15_SchG_assumption_derivation`
          ----- Column Name
          ----- Coloumn Source 
          ----- business_logic 
          ----- data_type
          ----- Is_FINAL_FLAG
            ----- assumption_effective_date
            ----- assumption_end_date
            ----- approved flag
            ----- entered by _______
            ----- timestamp here ... etc

            





In [0]:
%sql


CREATE OR REPLACE TEMP VIEW y15_assump_src AS
SELECT DISTINCT
--   y15_SchG_tier,
--   y15_SchG_line,
--   y15_SchG_weighting,
--   y15_SchG_present,
--   y15_SchG_rollup,
  y15_class,
--   y15_SchG_class_code,
--   balance_type,
  maturity_bucket,
--   maturity_column,
  Fse_Flag,
  Wholesale_Flag,
  Counterparty_Type,
  Ops_flag,
  Collateral_Flag,
  Deposit_Type,
  Product_Type,
  internal_elim_flag,
  Bdsp_Flag,
  legacy_bank
FROM finance.reporting.depositsplus_eom;

SELECT * FROM y15_assump_src;


In [0]:
%sql
-- Y15 assumption combinations with effective/end dates and who uploaded them
CREATE TABLE IF NOT EXISTS finance.liquidity.y15_assump (
  y15_class          STRING,
  maturity_bucket    STRING,
  fse_flag           STRING,
  wholesale_flag     STRING,
  counterparty_type  STRING,
  ops_flag           STRING,
  collateral_flag    STRING,
  deposit_type       STRING,
  product_type       STRING,
  internal_elim_flag STRING,
  bdsp_flag          STRING,
  legacy_bank        STRING,
  effective_date     DATE      COMMENT 'Date the combination takes effect',
  end_date           DATE      COMMENT 'Last date the combination applies; NULL = still active',
  active_flag        STRING    COMMENT 'Y/N',
  uploaded_by        STRING    COMMENT 'User who uploaded the row',
  uploaded_ts        TIMESTAMP COMMENT 'When the row was uploaded',
  modified_by        STRING,
  modified_ts        TIMESTAMP
)
USING DELTA;

-- Add combinations that are not already active (null-safe match on every column, so re-running adds nothing new)
MERGE INTO finance.liquidity.y15_assump AS tgt
USING y15_assump_src AS src
ON  tgt.active_flag = 'Y'
AND tgt.y15_class          <=> src.y15_class
AND tgt.maturity_bucket    <=> src.maturity_bucket
AND tgt.fse_flag           <=> src.Fse_Flag
AND tgt.wholesale_flag     <=> src.Wholesale_Flag
AND tgt.counterparty_type  <=> src.Counterparty_Type
AND tgt.ops_flag           <=> src.Ops_flag
AND tgt.collateral_flag    <=> src.Collateral_Flag
AND tgt.deposit_type       <=> src.Deposit_Type
AND tgt.product_type       <=> src.Product_Type
AND tgt.internal_elim_flag <=> src.internal_elim_flag
AND tgt.bdsp_flag          <=> src.Bdsp_Flag
AND tgt.legacy_bank        <=> src.legacy_bank
WHEN NOT MATCHED THEN INSERT (
  y15_class, maturity_bucket, fse_flag, wholesale_flag, counterparty_type, ops_flag, collateral_flag,
  deposit_type, product_type, internal_elim_flag, bdsp_flag, legacy_bank,
  effective_date, end_date, active_flag, uploaded_by, uploaded_ts, modified_by, modified_ts
) VALUES (
  src.y15_class, src.maturity_bucket, src.Fse_Flag, src.Wholesale_Flag, src.Counterparty_Type, src.Ops_flag, src.Collateral_Flag,
  src.Deposit_Type, src.Product_Type, src.internal_elim_flag, src.Bdsp_Flag, src.legacy_bank,
  current_date(), CAST(NULL AS DATE), 'Y', current_user(), current_timestamp(), current_user(), current_timestamp()
);

SELECT effective_date, end_date, active_flag, uploaded_by, COUNT(1) AS row_count
FROM finance.liquidity.y15_assump
GROUP BY ALL
ORDER BY effective_date DESC;

In [0]:
%sql
select * from finance.liquidity.y15_assump 



In [0]:
%sql
SELECT DISTINCT y15_class FROM finance.reporting.depositsplus_eom 

-- 17(0000) -- 

-- 0 - snv or 1 -pnfp
-- 3 or 9 or retail or wholesale

-- 1 or 0 for OPS OR NOT

-- 1-5 FOR different counterparties

-- D (DDA), OR T (TIME)

-- A TO E for maturity

-- 171903DA


In [0]:
%sql
-- One row per derived column: where it comes from, the business rule, type, and approval/audit info.
-- Sample loaded from the mv_2 step (CELL 13 of the deposits plus build).
CREATE TABLE IF NOT EXISTS finance.liquidity.y15_SchG_assumption_derivation (
  step_name                 STRING    COMMENT 'Build step that creates the column, e.g. mv_2',
  column_name               STRING    COMMENT 'Output column name',
  column_source             STRING    COMMENT 'Source table(s) and column(s)',
  business_logic            STRING    COMMENT 'Rule in plain words / SQL',
  data_type                 STRING,
  is_final_flag             STRING    COMMENT 'Y = used in the final Y15 Schedule G output',
  assumption_effective_date DATE,
  assumption_end_date       DATE      COMMENT 'NULL = still in effect',
  approved_flag             STRING    COMMENT 'Y/N',
  entered_by                STRING,
  entered_ts                TIMESTAMP,
  modified_by               STRING,
  modified_ts               TIMESTAMP
)
USING DELTA;

CREATE OR REPLACE TEMP VIEW y15_derivation_src AS
SELECT * FROM VALUES
  ('mv_2', 'maturity_bucket', 'v1_table.days_to_maturity',
   'LT 30 Days if days_to_maturity <= 30 or NULL; 31-90 Days; 91-180 Days; 181-365 Days; else GT 365 Days', 'STRING', 'Y'),
  ('mv_2', 'maturity_flag_short', 'v1_table.days_to_maturity',
   'LT30 if days_to_maturity <= 30 or NULL, else GT30', 'STRING', 'N'),
  ('mv_2', 'nmd_time_flag', 'v1_table.deposit_type',
   'TIME if deposit_type in (TIME, CDs, Brokered), else NMD', 'STRING', 'N'),
  ('mv_2', 'wholesale_flag', 'odm_raw_results.wholesale_client_type; v1_table.ac_ar_id',
   'Y if wholesale_client_type = Wholesale and ac_ar_id not in (16501007815721, 16501007815713), else N', 'STRING', 'Y'),
  ('mv_2', 'ops_flag', 'odm_raw_results.op_balance',
   'Y if ODM op_balance > 0.01, else N', 'STRING', 'Y'),
  ('mv_2', 'fse_flag', 'v1_table.naics_code',
   'Y if first 2 digits of naics_code = 52 (finance and insurance), else N', 'STRING', 'Y'),
  ('mv_2', 'op_balance', 'odm_raw_results.op_balance, ledger_balance; v1_table.ledger_balance',
   'ODM op_balance / ODM ledger_balance * account ledger_balance (TRY_DIVIDE, NULL if ODM ledger = 0)', 'DOUBLE', 'Y'),
  ('mv_2', 'non_op_balance', 'odm_raw_results.op_balance, ledger_balance; v1_table.ledger_balance',
   'account ledger_balance - op_balance', 'DOUBLE', 'Y'),
  ('mv_2', 'internal_elim_flag', 'v1_table.tax_id, account_number',
   'BankElim for tax_id 464094083/560989633 or listed bank accounts; HoldcoElim for tax_id 581625031/621812853/934843078 or listed holdco accounts; else N', 'STRING', 'Y')
AS t(step_name, column_name, column_source, business_logic, data_type, is_final_flag)
UNION ALL
-- Management reporting levels 1-9 from the entity hierarchy
SELECT 'mv_2', concat('mgmt_reporting_lvl_', n),
       concat('d_entity_hierarchy_wide.Mgmt_Rept_LVL0', n, '_description'),
       concat('Entity hierarchy level ', n, ' description, joined on entity_id = v1_table.entity'), 'STRING', 'N'
FROM (SELECT explode(sequence(1, 9)) AS n);

-- Add new columns only (re-running does not duplicate)
MERGE INTO finance.liquidity.y15_SchG_assumption_derivation AS tgt
USING y15_derivation_src AS src
ON tgt.step_name = src.step_name AND tgt.column_name = src.column_name AND tgt.assumption_end_date IS NULL
WHEN NOT MATCHED THEN INSERT (
  step_name, column_name, column_source, business_logic, data_type, is_final_flag,
  assumption_effective_date, assumption_end_date, approved_flag, entered_by, entered_ts, modified_by, modified_ts
) VALUES (
  src.step_name, src.column_name, src.column_source, src.business_logic, src.data_type, src.is_final_flag,
  current_date(), CAST(NULL AS DATE), 'N', current_user(), current_timestamp(), current_user(), current_timestamp()
);

SELECT * FROM finance.liquidity.y15_SchG_assumption_derivation ORDER BY step_name, column_name;